# 04 — Geometry (`geometry.py`)

This notebook isolates the coordinate language used later by invariant point attention and geometric losses: quaternions, rotations, translations, homogeneous transforms, local frames, and frame-independent comparison.

## Stage map

```text
quaternion q -----------------> rotation matrix R
                                      |
translation t ------------------------+--> transform T = [R | t]
                                                        |
                                      +-----------------+----------------+
                                      |                                  |
                                apply(T, point)                    invert(T)
                                      |                                  |
                                      +---- local <-> global coordinates-+

backbone N, CA, C --> one local frame per residue
predicted vs teacher points --> Kabsch alignment --> frame-independent RMSD
```

Geometry is a toolbox used by the structure module and losses; it is not another neural-network trunk stage.

In [ ]:
import sys
import torch
import matplotlib.pyplot as plt

sys.path.insert(0, "../src")  # package source lives one level up
torch.manual_seed(0)
plt.rcParams["figure.figsize"] = (8, 4)

## 1. Rotations, translations, and rigid transforms

A quaternion is normalized and converted into a 3×3 rotation matrix `R`. Combining `R` with a translation `t` gives a 4×4 transform `T = [R|t]`. `apply` maps local points to global coordinates and `invert` maps them back.

In [ ]:
from af2_from_scratch.geometry import (
    quat_to_rot,
    make_T,
    apply,
    invert,
    frames_from_backbone,
)

R = quat_to_rot(
    torch.tensor([[1.0, 0.3, -0.2, 0.1]])
)  # unnormalized quaternion -> rotation
T = make_T(R, torch.tensor([[1.0, 2.0, 3.0]]))
x = torch.tensor([[0.0, 0.0, 1.5]])
print("local -> global:", apply(T, x))
print("and back:      ", apply(invert(T), apply(T, x)))  # round-trip recovers x

## 2. Local residue frames

A local frame gives every residue its own coordinate system. `frames_from_backbone` places the origin at Cα, points one axis toward C, orthogonalizes another toward N, and obtains the third with a cross product. Comparing structures in these local coordinates removes arbitrary global motion.

In [ ]:
N = torch.tensor([[0.0, 1.0, 0.0], [1.0, 1.0, 0.0]])
CA = torch.tensor([[0.0, 0.0, 0.0], [1.0, 0.0, 0.0]])
C = torch.tensor([[1.0, 0.0, 0.0], [2.0, 0.0, 0.0]])
frames = frames_from_backbone(N, CA, C)
print("residue frames:", tuple(frames.shape))
print("frame origins equal CA:", torch.allclose(frames[..., :3, 3], CA))

## 3. Kabsch RMSD

Two identical folds may be translated or rotated differently. Kabsch alignment finds the best rigid alignment before calculating RMSD, so evaluation measures the fold instead of the choice of global coordinate system.

In [ ]:
from af2_from_scratch.geometry import kabsch_rmsd

points = torch.tensor([[0.0, 0.0, 0.0], [1.0, 0.0, 0.0], [1.0, 1.0, 0.0]])
rotation = quat_to_rot(torch.tensor([1.0, 0.0, 0.0, 0.5]))
moved = points @ rotation + torch.tensor([4.0, -2.0, 3.0])
print("raw RMSD:    ", (points - moved).norm(dim=-1).pow(2).mean().sqrt().item())
print("Kabsch RMSD: ", kabsch_rmsd(points, moved).item())

**Next:** `05_structure_module.ipynb` uses these transforms to iteratively place and orient residues.